In [ ]:
# Notebook 01 - Carga y limpieza de datos

## Objetivo

Importar los datos financieros obtenidos de Economatica y realizar el preprocesamiento inicial necesario para el análisis.

## Entradas

- Series históricas de precios.
- Bases de datos financieras originales.

## Tareas principales

1. Cargar los datos originales.
2. Estandarizar formatos y fechas.
3. Filtrar emisoras con suficiente información histórica.
4. Construir una base de datos limpia.

## Salidas

- Series de precios procesadas.
- Universo de activos filtrado.

## Siguiente paso

Los resultados generados en este notebook se utilizan en el Notebook 02 para calcular los rendimientos logarítmicos.


# =========================================
# Importar librerías principales
# =========================================

# pandas:
# Manejo de tablas y bases de datos
import pandas as pd

# numpy:
# Operaciones numéricas y matriciales
import numpy as np

# pathlib:
# Manejo profesional de rutas y carpetas
from pathlib import Path


# =========================================
# Definir directorio principal del proyecto
# =========================================

# Ruta raíz de la tesis
BASE_DIR = Path(r"C:\MRGB\Personal\Tesis")


# =========================================
# Definir carpetas importantes
# =========================================

# Carpeta donde están las bases de datos
DATA_DIR = BASE_DIR / "Datas"

# Carpeta donde guardaremos resultados
OUTPUT_DIR = BASE_DIR / "outputs"
# Crear carpeta outputs si no existe
OUTPUT_DIR.mkdir(exist_ok=True)
# Crear carpeta de datos procesados
(OUTPUT_DIR / "datos_procesados").mkdir(exist_ok=True)


# =========================================
# Definir archivo principal de precios
# =========================================

# Base de datos principal de Economática
archivo_precios = DATA_DIR / "02 precios_diarios_ajustados_1994_2026_activas.csv"


# =========================================
# Cargar archivo CSV
# =========================================

# Leer la base de datos
df = pd.read_csv(archivo_precios)


# =========================================
# Exploración inicial de la base
# =========================================

# Mostrar dimensiones:
# (filas, columnas)
print("Dimensiones:")
print(df.shape)

# Mostrar primeras filas
print("\nPrimeras filas:")
display(df.head())

# =========================================
# Mostrar nombres de columnas
# =========================================

for col in df.columns:
    print(col)

# =========================================
# Limpieza inicial de nombres de columnas
# =========================================

# Hacemos una copia para no alterar el dataframe original
precios = df.copy()

# Renombramos las primeras columnas importantes
# La primera columna identifica el activo base de Economática
# La segunda columna contiene la fecha
precios = precios.rename(columns={
    precios.columns[0]: "Activo",
    precios.columns[1]: "Fecha"
})

# Limpiamos nombres de columnas:
# Si una columna contiene "|", nos quedamos con lo que está después del último "|"
# Ejemplo:
# "Cierre|ajust p/var cap|en moneda orig|ACCELSAB" -> "ACCELSAB"
nuevas_columnas = []

for col in precios.columns:
    if "|" in col:
        nuevas_columnas.append(col.split("|")[-1])
    else:
        nuevas_columnas.append(col)

precios.columns = nuevas_columnas

# Mostramos las primeras columnas ya limpias
print(precios.columns[:20])
display(precios.head())

# =========================================
# Depuración de columnas y conversión de fechas
# =========================================

# Eliminamos columnas que no corresponden a activos financieros
# En este caso, solo nos quedamos con "Fecha" y las columnas de tickers
columnas_a_eliminar = ["Activo", "en moneda orig"]

precios = precios.drop(columns=columnas_a_eliminar, errors="ignore")

# Convertimos la columna Fecha a formato de fecha reconocido por Python
precios["Fecha"] = pd.to_datetime(precios["Fecha"], errors="coerce")

# Ordenamos la base por fecha
precios = precios.sort_values("Fecha")

# Revisamos resultado
print("Dimensiones después de depurar columnas:")
print(precios.shape)

print("\nPeriodo disponible:")
print(precios["Fecha"].min(), "a", precios["Fecha"].max())

display(precios.head())

# =========================================
# Filtrar periodo 2010-2025 y convertir datos numéricos
# =========================================

# Nos quedamos únicamente con observaciones desde 2010
precios = precios[
    (precios["Fecha"] >= "2010-01-01") &
    (precios["Fecha"] <= "2025-12-31")
]

# Reemplazar guiones "-" por valores faltantes
precios = precios.replace("-", np.nan)

# Lista de columnas numéricas
# Excluimos la columna Fecha
columnas_precios = precios.columns.drop("Fecha")

# Convertimos todas las columnas de precios a tipo numérico
for col in columnas_precios:
    precios[col] = pd.to_numeric(precios[col], errors="coerce")

# Revisamos resultado
print("Dimensiones finales:")
print(precios.shape)

print("\nPeriodo filtrado:")
print(precios["Fecha"].min(), "a", precios["Fecha"].max())

display(precios.head())

# =========================================
# Analizar valores faltantes por activo
# =========================================

# Contar valores faltantes por columna
faltantes = precios.isna().sum()

# Convertir a dataframe para visualizar mejor
faltantes_df = pd.DataFrame({
    "Activo": faltantes.index,
    "Valores_Faltantes": faltantes.values
})

# Ordenar de mayor a menor cantidad de faltantes
faltantes_df = faltantes_df.sort_values(
    by="Valores_Faltantes",
    ascending=False
)

# Mostrar resultados
display(faltantes_df.head(20))

# =========================================
# Analizar valores faltantes por activo
# =========================================

# Contamos cuántos valores faltantes tiene cada columna
faltantes = precios.isna().sum()

# Convertimos el resultado a una tabla para revisarlo mejor
faltantes_df = pd.DataFrame({
    "Activo": faltantes.index,
    "Valores_Faltantes": faltantes.values
})

# Quitamos la columna Fecha porque no es un activo financiero
faltantes_df = faltantes_df[faltantes_df["Activo"] != "Fecha"]

# Calculamos el porcentaje de datos faltantes respecto al total de fechas
faltantes_df["Porcentaje_Faltante"] = (
    faltantes_df["Valores_Faltantes"] / len(precios)
) * 100

# Ordenamos de mayor a menor cantidad de faltantes
faltantes_df = faltantes_df.sort_values(
    by="Porcentaje_Faltante",
    ascending=False
)

# Mostramos los 20 activos con más datos faltantes
display(faltantes_df.head(20))

# =========================================
# Seleccionar activos con suficiente información
# =========================================

# Definir porcentaje mínimo de datos requeridos
umbral = 0.80

# Calcular porcentaje de datos NO faltantes
porcentaje_disponible = 1 - (
    precios.isna().sum() / len(precios)
)

# Seleccionar activos que cumplen el criterio
activos_validos = porcentaje_disponible[
    porcentaje_disponible >= umbral
].index.tolist()

# Mantener Fecha y activos válidos
columnas_finales = ["Fecha"] + [
    col for col in activos_validos if col != "Fecha"
]

# Crear base depurada
precios_limpios = precios[columnas_finales]

# Mostrar dimensiones finales
print("Dimensiones originales:")
print(precios.shape)

print("\nDimensiones después de limpieza:")
print(precios_limpios.shape)

print("\nNúmero de activos conservados:")
print(len(precios_limpios.columns) - 1)

display(precios_limpios.head())

# =========================================================
# Guardar base de datos limpia
# =========================================================

# Crear nombre del archivo final limpio
archivo_salida = OUTPUT_DIR / "datos_procesados" / "precios_limpios_al_80pct.csv"

# Guardar DataFrame limpio en CSV
precios_limpios.to_csv(archivo_salida, index=False)

# Confirmación en pantalla
print("Base limpia guardada correctamente.")
print(f"Ruta de salida: {archivo_salida}")

# =========================================================
# Resumen final del Notebook 01
# =========================================================

print("Resumen de limpieza")
print("-------------------")
print(f"Periodo final: {precios_limpios['Fecha'].min()} a {precios_limpios['Fecha'].max()}")
print(f"Número de observaciones temporales: {precios_limpios.shape[0]}")
print(f"Número de activos conservados: {precios_limpios.shape[1] - 1}")
print(f"Umbral mínimo de disponibilidad utilizado: 80%")
print(f"Archivo generado: {archivo_salida}")

Dimensiones:
(8985, 199)

Primeras filas:


,Activo,Fecha,Cierre|ajust p/var cap|en moneda orig,Cierre|ajust p/var cap|en moneda orig|ACCELSAB,Cierre|ajust p/var cap|en moneda orig|GAVA,Cierre|ajust p/var cap|en moneda orig|GAVB,Cierre|ajust p/var cap|en moneda orig|GAVEC001,Cierre|ajust p/var cap|en moneda orig|AGUILASA,Cierre|ajust p/var cap|en moneda orig|AGUILASB,Cierre|ajust p/var cap|en moneda orig|AGUILASCPO,...,Cierre|ajust p/var cap|en moneda orig|URBI,Cierre|ajust p/var cap|en moneda orig|VALUEGFO,Cierre|ajust p/var cap|en moneda orig|VASCONI,Cierre|ajust p/var cap|en moneda orig|VESTA,Cierre|ajust p/var cap|en moneda orig|VINTE,Cierre|ajust p/var cap|en moneda orig|VISTAA,Cierre|ajust p/var cap|en moneda orig|VISTAC,Cierre|ajust p/var cap|en moneda orig|VITROA,Cierre|ajust p/var cap|en moneda orig|VOLARA,Cierre|ajust p/var cap|en moneda orig|WALMEX
0,AUTLANB<XMEX>,1991-12-02,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.46943684262
1,AUTLANB<XMEX>,1991-12-03,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.46943684262
2,AUTLANB<XMEX>,1991-12-04,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.46817830001
3,AUTLANB<XMEX>,1991-12-05,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.46817830001
4,AUTLANB<XMEX>,1991-12-06,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.46188558697


Activo
Fecha
Cierre|ajust p/var cap|en moneda orig
Cierre|ajust p/var cap|en moneda orig|ACCELSAB
Cierre|ajust p/var cap|en moneda orig|GAVA
Cierre|ajust p/var cap|en moneda orig|GAVB
Cierre|ajust p/var cap|en moneda orig|GAVEC001
Cierre|ajust p/var cap|en moneda orig|AGUILASA
Cierre|ajust p/var cap|en moneda orig|AGUILASB
Cierre|ajust p/var cap|en moneda orig|AGUILASCPO
Cierre|ajust p/var cap|en moneda orig|AGUILASD
Cierre|ajust p/var cap|en moneda orig|AGUILASL
Cierre|ajust p/var cap|en moneda orig|ALEATIC
Cierre|ajust p/var cap|en moneda orig|ALPEKA
Cierre|ajust p/var cap|en moneda orig|ALSEA
Cierre|ajust p/var cap|en moneda orig|ALTERNAB
Cierre|ajust p/var cap|en moneda orig|AMXA
Cierre|ajust p/var cap|en moneda orig|AMXB
Cierre|ajust p/var cap|en moneda orig|AMXL
Cierre|ajust p/var cap|en moneda orig|ARA
Cierre|ajust p/var cap|en moneda orig|AC
Cierre|ajust p/var cap|en moneda orig|ARISTOSA
Cierre|ajust p/var cap|en moneda orig|ASURB
Cierre|ajust p/var cap|en moneda orig|AUTLANB
C

,Activo,Fecha,en moneda orig,ACCELSAB,GAVA,GAVB,GAVEC001,AGUILASA,AGUILASB,AGUILASCPO,...,URBI,VALUEGFO,VASCONI,VESTA,VINTE,VISTAA,VISTAC,VITROA,VOLARA,WALMEX
0,AUTLANB<XMEX>,1991-12-02,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.46943684262
1,AUTLANB<XMEX>,1991-12-03,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.46943684262
2,AUTLANB<XMEX>,1991-12-04,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.46817830001
3,AUTLANB<XMEX>,1991-12-05,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.46817830001
4,AUTLANB<XMEX>,1991-12-06,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.46188558697


Dimensiones después de depurar columnas:
(8985, 197)

Periodo disponible:
1991-12-02 00:00:00 a 2026-05-08 00:00:00


...

,Activo,Valores_Faltantes
4,GAVEC001,4174
3,GAVB,4174
8,AGUILASD,4174
6,AGUILASB,4174
5,AGUILASA,4174
50,DIABLOSA,4174
9,AGUILASL,4174
132,LACOMERUB,4174
155,PMCPACC,4174
184,TPLAY_00125,4174


,Activo,Valores_Faltantes,Porcentaje_Faltante
4,GAVEC001,4174,100.0
3,GAVB,4174,100.0
6,AGUILASB,4174,100.0
5,AGUILASA,4174,100.0
50,DIABLOSA,4174,100.0
9,AGUILASL,4174,100.0
8,AGUILASD,4174,100.0
155,PMCPACC,4174,100.0
184,TPLAY_00125,4174,100.0
183,TPLAY,4174,100.0


Dimensiones originales:
(4174, 197)

Dimensiones después de limpieza:
(4174, 46)

Número de activos conservados:
45


,Fecha,ALPEKA,ALSEA,AMXB,ARA,AC,ASURB,AUTLANB,AXTELCPO,BACHOCOB,...,PINFRA,Q,RA,SIGMAFA,SIMECB,SORIANAB,SPORTS,TLEVISACPO,VESTA,WALMEX
4719,2010-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4720,2010-01-04,NaN,8.727598,10.649955,6.583465,20.500964,42.587085,21.452708,12.61,19.55927,...,22.312959,NaN,NaN,4.650864,34.636666,29.707334,NaN,45.989794,NaN,19.474547
4721,2010-01-05,NaN,8.736154,10.594917,6.562043,20.505678,42.761105,21.452708,12.79,19.55927,...,22.305318,NaN,NaN,4.579599,34.814098,30.228034,NaN,45.364887,NaN,19.857106
4722,2010-01-06,NaN,8.727598,10.549051,6.583465,20.411399,43.733221,21.987267,12.93,NaN,...,22.924273,NaN,NaN,4.614699,33.973629,30.602573,NaN,44.934208,NaN,19.857106
4723,2010-01-07,NaN,8.676259,10.677475,6.540622,20.500964,43.937246,22.296749,12.88,19.55927,...,22.916632,NaN,NaN,4.588108,32.871681,31.059328,NaN,45.643562,NaN,19.490896


Base limpia guardada correctamente.
Ruta de salida: C:\MRGB\Personal\Tesis\outputs\datos_procesados\precios_limpios_al_80pct.csv
Resumen de limpieza
-------------------
Periodo final: 2010-01-01 00:00:00 a 2025-12-31 00:00:00
Número de observaciones temporales: 4174
Número de activos conservados: 45
Umbral mínimo de disponibilidad utilizado: 80%
Archivo generado: C:\MRGB\Personal\Tesis\outputs\datos_procesados\precios_limpios_al_80pct.csv
